In [1]:
from find_missing_jb import find_missing_entry, find_missing_output_string
import jailbreak_langgraph_trace
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from judge_action_output import judge_selective_jailbreak
import re
import json

def sort_json_by_name(json_data):
    """
    Sort JSON objects by their 'name' key in format 'action_X_jb_prompt_Y'.
    Sorts numerically by action index first, then by jb_prompt index.
    
    Args:
        json_data: List of JSON objects with 'name' keys
    
    Returns:
        Sorted list of JSON objects
    """
    def extract_numbers(name):
        """Extract action and jb_prompt numbers from name for proper sorting"""
        match = re.match(r'action_(\d+)_jb_prompt_(\d+)', name)
        if match:
            action_num = int(match.group(1))
            jb_prompt_num = int(match.group(2))
            return (action_num, jb_prompt_num)
        else:
            # If name doesn't match expected format, sort to end
            return (float('inf'), float('inf'))
    
    # Sort by action number first, then by jb_prompt number
    sorted_data = sorted(json_data, key=lambda x: extract_numbers(x.get('name', '')))
    
    return sorted_data

load_dotenv()

data_dir = './data/action_jb_result'
jailbreak_result_files = [f for f in os.listdir(data_dir) if f.endswith('.json')]

for filename in jailbreak_result_files:

    all_missing_jb = []

    input_file = f"data/action_jb_result/{filename}"

    missing_entries = find_missing_entry(filename=input_file)
    all_missing_jb.extend(missing_entries)

    # missing_output_strings = find_missing_output_string(filename=input_file)
    # all_missing_jb.extend(missing_output_strings)

    print(f"filename: {filename}")
    print("all_missing_jb: ", all_missing_jb)

    gpt_oss_low_reasoning = ChatGroq(
        model="openai/gpt-oss-20b", 
        temperature=0.2, 
        api_key=os.getenv("GROQ_API_KEY"),
        reasoning_format="parsed",
        reasoning_effort="low"
    )
    # Determine injection type based on filename
    if "human" in filename:
        injection_type = "human_injection"
    elif "ai" in filename:
        injection_type = "ai_injection"
    elif "tool" in filename:
        injection_type = "tool_injection"
    else:
        injection_type = "human_injection"  # default fallback

    jailbreak_langgraph_trace.execute_selective_jailbreak_evaluation(
        graph_file="data/traces/detailed_graph_langgraph_multi_trace.json",
        model=gpt_oss_low_reasoning,
        jailbreak_prompt_file="data/PAIR_jailbreaking/successful_PAIR_prompt_gpt_oss_20b.json",
        target_names=all_missing_jb,
        output_file=input_file,  # Custom output file
        injection_type=injection_type,
        intermediary_phrase=""  # No intermediary
    )


    with open(input_file, 'r') as f:
        data = json.load(f)

    # Sort the data
    sorted_data = sort_json_by_name(data)

    # Save back to file (or new file)
    with open(input_file, 'w') as f:
        json.dump(sorted_data, f, indent=2)


    all_missing_human_judge = []
    missing_entries = find_missing_entry(filename=f"data/action_judge_result/{filename.replace('jailbreak', 'judge')}")
    all_missing_human_judge.extend(missing_entries)

    jb_input_file = input_file
    output_file = f"data/action_judge_result/{filename.replace('jailbreak', 'judge')}"    
    jailbreak_prompt_file = "data/PAIR_jailbreaking/successful_PAIR_prompt_gpt_oss_20b.json"

    model_name = "gpt-4o-mini"  # Change this to specify which model to use

    print("all_missing_human_judge: ", all_missing_human_judge)

    judge_selective_jailbreak(jb_input_file, output_file, jailbreak_prompt_file, all_missing_human_judge, model_name, False)

    with open(output_file, 'r') as f:
        data = json.load(f)

    # Sort the data
    sorted_data = sort_json_by_name(data)

    # Save back to file (or new file)
    with open(output_file, 'w') as f:
        json.dump(sorted_data, f, indent=2)


Processing file: data/action_jb_result/jailbreak_results_human_gpt_oss_20b.json
Action range: 0 to 28
✓ No missing actions found

Checking action_0:

Checking action_1:

Checking action_2:

Checking action_3:

Checking action_4:

Checking action_5:

Checking action_6:

Checking action_7:

Checking action_8:

Checking action_9:

Checking action_10:

Checking action_11:

Checking action_12:

Checking action_13:

Checking action_14:

Checking action_15:

Checking action_16:

Checking action_17:

Checking action_18:

Checking action_19:

Checking action_20:

Checking action_21:

Checking action_22:

Checking action_23:

Checking action_24:

Checking action_25:

Checking action_26:

Checking action_27:

Checking action_28:

Summary for data/action_jb_result/jailbreak_results_human_gpt_oss_20b.json:
Total missing individual entries: 0
Total missing action entries (complete actions): 0
Total combined missing entries: 0
filename: jailbreak_results_human_gpt_oss_20b.json
all_missing_jb:  []
No

# Final check

In [2]:
import os
from find_missing_jb import find_missing_entry

data_dir = './data/action_jb_result'
jailbreak_result_files = [f for f in os.listdir(data_dir) if f.endswith('.json')]

for filename in jailbreak_result_files:
    print(f"\nChecking jailbreak file: {filename}")
    
    # Check missing entries in jailbreak results
    jb_input_file = f"data/action_jb_result/{filename}"
    missing_jb_entries = find_missing_entry(filename=jb_input_file)
    # missing_jb_output_strings = find_missing_output_string(filename=jb_input_file)
    
    print(f"Missing jailbreak entries: {missing_jb_entries}")
    # print(f"Missing jailbreak output strings: {missing_jb_output_strings}")
    
    # Check missing entries in judge results
    judge_filename = filename.replace('jailbreak', 'judge')
    judge_input_file = f"data/action_judge_result/{judge_filename}"
    
    try:
        missing_judge_entries = find_missing_entry(filename=judge_input_file)
        print(f"Missing judge entries: {missing_judge_entries}")
    except FileNotFoundError:
        print(f"Judge file not found: {judge_filename}")



Checking jailbreak file: jailbreak_results_human_gpt_oss_20b.json

Processing file: data/action_jb_result/jailbreak_results_human_gpt_oss_20b.json
Action range: 0 to 28
✓ No missing actions found

Checking action_0:

Checking action_1:

Checking action_2:

Checking action_3:

Checking action_4:

Checking action_5:

Checking action_6:

Checking action_7:

Checking action_8:

Checking action_9:

Checking action_10:

Checking action_11:

Checking action_12:

Checking action_13:

Checking action_14:

Checking action_15:

Checking action_16:

Checking action_17:

Checking action_18:

Checking action_19:

Checking action_20:

Checking action_21:

Checking action_22:

Checking action_23:

Checking action_24:

Checking action_25:

Checking action_26:

Checking action_27:

Checking action_28:

Summary for data/action_jb_result/jailbreak_results_human_gpt_oss_20b.json:
Total missing individual entries: 0
Total missing action entries (complete actions): 0
Total combined missing entries: 0
Missin

In [4]:
import os
import json
from find_missing_jb import find_missing_entry

data_dir = './data/action_jb_result'
jailbreak_result_files = [f for f in os.listdir(data_dir) if f.endswith('.json')]

for filename in jailbreak_result_files:
    print(f"\nChecking jailbreak file: {filename}")
    
    # Count entries in jailbreak results
    jb_input_file = f"data/action_jb_result/{filename}"
    with open(jb_input_file, 'r') as f:
        jb_data = json.load(f)
    jb_count = len(jb_data)
    
    # missing_jb_entries = find_missing_entry(filename=jb_input_file)
    # # missing_jb_output_strings = find_missing_output_string(filename=jb_input_file)
    
    print(f"Jailbreak entries count ({jb_count})")
    # print(f"Missing jailbreak entries ({len(missing_jb_entries)}): {missing_jb_entries}")
    # print(f"Missing jailbreak output strings: {missing_jb_output_strings}")
    
    # Count entries in judge results
    judge_filename = filename.replace('jailbreak', 'judge')
    judge_input_file = f"data/action_judge_result/{judge_filename}"
    
    try:
        with open(judge_input_file, 'r') as f:
            judge_data = json.load(f)
        judge_count = len(judge_data)
        
        # missing_judge_entries = find_missing_entry(filename=judge_input_file)
        print(f"Judge entries count ({judge_count})")
        # print(f"Missing judge entries ({len(missing_judge_entries)}): {missing_judge_entries}")
    except FileNotFoundError:
        print(f"Judge file not found: {judge_filename}")



Checking jailbreak file: jailbreak_results_human_gpt_oss_20b.json
Jailbreak entries count (435)
Judge entries count (435)

Checking jailbreak file: jailbreak_results_tool_gpt_oss_20b.json
Jailbreak entries count (300)
Judge entries count (300)

Checking jailbreak file: jailbreak_results_ai_w_intermediary_gpt_oss_20b.json
Jailbreak entries count (435)
Judge entries count (435)

Checking jailbreak file: jailbreak_results_tool_w_intermediary_gpt_oss_20b.json
Jailbreak entries count (300)
Judge entries count (300)

Checking jailbreak file: jailbreak_results_ai_gpt_oss_20b.json
Jailbreak entries count (435)
Judge entries count (435)

Checking jailbreak file: jailbreak_results_human_w_intermediary_gpt_oss_20b.json
Jailbreak entries count (435)
Judge entries count (435)
